# SurfaceTarget-CRC — phase 01 — patch 1 3

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.3
#
# TCGA RNA-seq File Inventory
# Gene Expression Quantification — STAR Counts
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import requests
import pandas as pd
import json

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

GDC_API = "https://api.gdc.cancer.gov"

PROJECTS = [
    "TCGA-COAD",
    "TCGA-READ"
]

WORKFLOW = "STAR - Counts"

# ------------------------------------------------------------
# 2. Construct GDC API filters
# ------------------------------------------------------------

filters = {
    "op": "and",
    "content": [
        {
            "op": "in",
            "content": {
                "field": "cases.project.project_id",
                "value": PROJECTS
            }
        },
        {
            "op": "in",
            "content": {
                "field": "files.data_type",
                "value": ["Gene Expression Quantification"]
            }
        },
        {
            "op": "in",
            "content": {
                "field": "files.analysis.workflow_type",
                "value": [WORKFLOW]
            }
        }
    ]
}

# ------------------------------------------------------------
# 3. Define metadata fields
# ------------------------------------------------------------

fields = [
    "file_id",
    "file_name",
    "file_size",
    "md5sum",
    "data_type",
    "access",
    "analysis.workflow_type",
    "cases.submitter_id",
    "cases.project.project_id",
    "cases.samples.submitter_id",
    "cases.samples.sample_type"
]

# ------------------------------------------------------------
# 4. Retrieve file metadata with pagination
# ------------------------------------------------------------

print("=" * 65)
print("SURFACETARGET-CRC")
print("TCGA RNA-SEQ FILE INVENTORY")
print("=" * 65)

all_files = []

offset = 0
page_size = 1000

while True:

    params = {
        "filters": json.dumps(filters),
        "fields": ",".join(fields),
        "format": "JSON",
        "size": page_size,
        "from": offset
    }

    response = requests.get(
        f"{GDC_API}/files",
        params=params,
        timeout=90
    )

    response.raise_for_status()

    payload = response.json()["data"]

    hits = payload.get("hits", [])

    total = payload["pagination"]["total"]

    all_files.extend(hits)

    print(
        f"Retrieved {len(all_files)} "
        f"of {total} files"
    )

    if not hits or len(all_files) >= total:
        break

    offset += len(hits)

print("\nMetadata retrieval completed.")

# ------------------------------------------------------------
# 5. Extract file and sample information
# ------------------------------------------------------------

records = []

for item in all_files:

    file_id = item.get("file_id")
    file_name = item.get("file_name")
    file_size = item.get("file_size")

    workflow = (
        item.get("analysis") or {}
    ).get("workflow_type")

    cases = item.get("cases") or []

    for case in cases:

        patient_id = case.get("submitter_id")

        project_id = (
            (case.get("project") or {})
            .get("project_id")
        )

        samples = case.get("samples") or []

        if not samples:

            records.append({
                "project": project_id,
                "patient_id": patient_id,
                "sample_id": None,
                "sample_type": "Unknown",
                "file_id": file_id,
                "file_name": file_name,
                "file_size_bytes": file_size,
                "workflow": workflow,
                "access": item.get("access"),
                "md5sum": item.get("md5sum")
            })

        else:

            for sample in samples:

                records.append({
                    "project": project_id,
                    "patient_id": patient_id,
                    "sample_id": sample.get("submitter_id"),
                    "sample_type": sample.get(
                        "sample_type", "Unknown"
                    ),
                    "file_id": file_id,
                    "file_name": file_name,
                    "file_size_bytes": file_size,
                    "workflow": workflow,
                    "access": item.get("access"),
                    "md5sum": item.get("md5sum")
                })

# ------------------------------------------------------------
# 6. Create metadata dataframe
# ------------------------------------------------------------

df = pd.DataFrame(records)

if df.empty:
    raise RuntimeError(
        "No STAR Counts metadata returned. "
        "Check the GDC query filters."
    )

# Remove exact duplicate metadata records
df = df.drop_duplicates()

print("\n" + "-" * 65)
print("RNA-SEQ METADATA SUMMARY")
print("-" * 65)

print(f"\nUnique RNA-seq files: {df['file_id'].nunique()}")

print(
    f"Unique patients: "
    f"{df['patient_id'].nunique()}"
)

print(
    f"Unique samples: "
    f"{df['sample_id'].nunique()}"
)

# ------------------------------------------------------------
# 7. Sample type distribution
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAMPLE TYPE DISTRIBUTION")
print("-" * 65)

sample_summary = (
    df.groupby(
        ["project", "sample_type"],
        dropna=False
    )
    .agg(
        unique_patients=("patient_id", "nunique"),
        unique_samples=("sample_id", "nunique"),
        unique_files=("file_id", "nunique")
    )
    .reset_index()
)

print(
    sample_summary.to_string(index=False)
)

# ------------------------------------------------------------
# 8. Workflow distribution
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("WORKFLOW DISTRIBUTION")
print("-" * 65)

print(
    df.groupby(
        ["project", "workflow"],
        dropna=False
    )["file_id"]
    .nunique()
    .to_string()
)

# ------------------------------------------------------------
# 9. Save inventory and metadata
# ------------------------------------------------------------

inventory_path = (
    METADATA_DIR / "tcga_rnaseq_file_inventory.csv"
)

summary_path = (
    METADATA_DIR / "tcga_rnaseq_sample_summary.csv"
)

df.to_csv(
    inventory_path,
    index=False
)

sample_summary.to_csv(
    summary_path,
    index=False
)

# Save retrieval information
retrieval_info = {
    "retrieved_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source": "NCI Genomic Data Commons",
    "endpoint": f"{GDC_API}/files",
    "projects": PROJECTS,
    "workflow": WORKFLOW,
    "data_type": "Gene Expression Quantification",
    "total_files_reported_by_api": total,
    "unique_files_retrieved": int(
        df["file_id"].nunique()
    ),
    "unique_patients": int(
        df["patient_id"].nunique()
    ),
    "unique_samples": int(
        df["sample_id"].nunique()
    ),
    "filters": filters,
    "fields": fields
}

retrieval_path = (
    METADATA_DIR / "tcga_rnaseq_retrieval_metadata.json"
)

with open(retrieval_path, "w") as f:
    json.dump(retrieval_info, f, indent=4)

# ------------------------------------------------------------
# 10. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FILES SAVED")
print("=" * 65)

print(f"\n1. {inventory_path.name}")
print(f"2. {summary_path.name}")
print(f"3. {retrieval_path.name}")

print("\nNo RNA-seq expression matrices downloaded.")

print("\nPATCH 1.3 COMPLETE")